# 🤖 Sprint 3: Machine Learning e a Descoberta do Risco
### Minicurso: Informática Biomédica Aplicada | Jornada InfoBio 2026

---

## Você chegou ao Sprint final — bem-vindo ao Machine Learning! 🚀🏆

No Sprint 1 você visualizou os dados. No Sprint 2 você os normalizou. Agora vamos usar tudo isso como combustível para um algoritmo real de inteligência artificial.

O **K-Means** vai analisar os 5 biomarcadores **ao mesmo tempo** e agrupar os atletas em clusters fisiológicos — **sem que ninguém tenha dito ao algoritmo o que procurar**. Esse tipo de análise, chamada de aprendizado não-supervisionado, é usada em hospitais e centros esportivos ao redor do mundo para identificar grupos de risco antes que o atleta sinta qualquer sintoma.

**🎯 Objetivo deste Sprint:**
> Aplicar o algoritmo **K-Means**, interpretar os clusters fisiológicos e realizar o **Desafio Master** (identificar o Atleta 05 — o perfil de Risco Silencioso nos dados).

---

## 🧠 O que é K-Means?

K-Means é um algoritmo que recebe uma pergunta simples:

> *"Dada uma nuvem de pontos, divida-a em K grupos. Cada ponto deve pertencer ao grupo cujo centro está mais próximo."*

Nós escolhemos **K = 3**, que no contexto esportivo pode representar:

| Grupo | Perfil Fisiológico | Decisão da Comissão Técnica |
|---|---|---|
| 🟢 Grupo 1 | Recuperado — marcadores normais | Pode jogar normalmente |
| 🟡 Grupo 2 | Fatigado — dano moderado | Treino leve, monitorar |
| 🔴 Grupo 3 | Alto Risco — estresse severo | Repouso obrigatório |

O algoritmo não sabe que esses grupos existem. Ele os **descobre sozinho** a partir dos biomarcadores.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Carregando os dados dos 22 atletas direto do GitHub
url = "https://raw.githubusercontent.com/FBRosito/jornada-infobio-2026/main/dados_atletas_minicurso.csv"
df = pd.read_csv(url)

colunas_bio = ['CK_UL', 'Cortisol_ugdL', 'LDH_UL', 'PCR_mgL', 'Testosterona_nmolL']
for col in colunas_bio:
    df[col + '_z'] = (df[col] - df[col].mean()) / df[col].std()

colunas_z = [c + '_z' for c in colunas_bio]
X = df[colunas_z].values

print('Dados preparados!')
print(f'Matriz de entrada para o K-Means: {X.shape[0]} atletas × {X.shape[1]} biomarcadores normalizados')

In [ ]:
# Rodando o K-Means com K=3
# random_state=42 garante que você obtenha o mesmo resultado toda vez que rodar
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X)

print('K-Means concluído! Distribuição dos atletas por cluster:')
print(df['Cluster'].value_counts().sort_index().rename({0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2'}))
print()
df[['ID_Atleta', 'Posicao', 'CK_UL', 'Cortisol_ugdL', 'Cluster']].sort_values('Cluster')

In [ ]:
plt.figure(figsize=(10, 7))

cores = {0: '#2196F3', 1: '#FF9800', 2: '#4CAF50'}
marcadores_cluster = {0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2'}

for cluster_id in sorted(df['Cluster'].unique()):
    subset = df[df['Cluster'] == cluster_id]
    plt.scatter(
        subset['CK_UL'],
        subset['Cortisol_ugdL'],
        c=cores[cluster_id],
        label=marcadores_cluster[cluster_id],
        s=140,
        edgecolors='white',
        linewidths=1.2,
        alpha=0.9
    )

plt.title('K-Means (k=3) — Agrupamento Fisiológico dos Atletas', fontsize=14, fontweight='bold')
plt.xlabel('CK — Creatina Quinase (U/L)\n[Dano Muscular Mecânico]', fontsize=11)
plt.ylabel('Cortisol (µg/dL)\n[Estresse Sistêmico]', fontsize=11)
plt.legend(title='Grupos Descobertos pela IA', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Perfil médio de cada cluster (Z-Score) — o 'átleta típico' de cada grupo
centroides = df.groupby('Cluster')[colunas_z].mean().round(2)
centroides.index = ['Cluster 0', 'Cluster 1', 'Cluster 2']
centroides.columns = ['CK', 'Cortisol', 'LDH', 'PCR', 'Testosterona']

print('Perfil médio de cada cluster (valores em Z-Score):')
print('Positivo (+) = acima da média do grupo | Negativo (−) = abaixo da média')
print()

plt.figure(figsize=(9, 4))
sns.heatmap(
    centroides,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn_r',
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Z-Score'}
)
plt.title('Heatmap dos Centroides — Perfil Fisiológico de Cada Cluster', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 🏆 DESAFIO MASTER

In [ ]:
# ✏️ SEU CÓDIGO AQUI
